In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    roc_auc_score
)

# ============================================================
# 1. LOAD DATASET
# ============================================================

df = pd.read_csv("/content/placement_predict_50k Dataset (1).csv")

print("Dataset Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

# ============================================================
# 2. TARGET VARIABLE
# ============================================================

# PlacementStatus is already:
# 0 = Not Placed
# 1 = Placed

y = df["PlacementStatus"]

# ============================================================
# 3. SELECT PREDICTOR VARIABLES
# ============================================================

features = [
    "Gender",
    "City",
    "CollegeTier",
    "Stream",
    "Specialisation",
    "Hostel",
    "HistoryOfBacklogs",

    "SGPA_Sem1",
    "SGPA_Sem2",
    "SGPA_Sem3",
    "SGPA_Sem4",
    "SGPA_Sem5",
    "SGPA_Sem6",
    "SGPA_Sem7",
    "SGPA_Sem8",

    "CGPA",
    "AttendancePercent",

    "Internships",
    "Projects",
    "Workshops",
    "Certifications",
    "Publications",

    "AptitudeTestScore",
    "SoftSkillsRating",
    "CodingTestScore",
    "MockInterviewScore",
    "ExtraCurricular"
]

#X = df[features].copy()
X = df[features]

# ============================================================
# 4. CATEGORICAL VARIABLES
# ============================================================

categorical_features = [
    "Gender",
    "City",
    "CollegeTier",
    "Stream",
    "Specialisation",
    "Hostel",
    "HistoryOfBacklogs"
]

# ============================================================
# 5. NUMERICAL VARIABLES
# ============================================================

numerical_features = [
    "SGPA_Sem1",
    "SGPA_Sem2",
    "SGPA_Sem3",
    "SGPA_Sem4",
    "SGPA_Sem5",
    "SGPA_Sem6",
    "SGPA_Sem7",
    "SGPA_Sem8",

    "CGPA",
    "AttendancePercent",

    "Internships",
    "Projects",
    "Workshops",
    "Certifications",
    "Publications",

    "AptitudeTestScore",
    "SoftSkillsRating",
    "CodingTestScore",
    "MockInterviewScore",
    "ExtraCurricular"
]

# ============================================================
# 6. HANDLE MISSING VALUES
# ============================================================

for col in numerical_features:
    X[col] = X[col].fillna(X[col].median())

for col in categorical_features:
    X[col] = X[col].fillna(X[col].mode()[0])

# ============================================================
# 7. TRAIN-TEST SPLIT
# ============================================================
#stratify=y = Keep the same proportion of values in both training and testing splits.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# ============================================================
# 8. PREPROCESSING
# ============================================================
#ColumnTransformer allows us to preprocess different types of features differently and combine the results into a single feature matrix

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            numerical_features
        ),

        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore",
                drop="first"
            ),
            categorical_features
        )
    ]
)

# ============================================================
# 9. BINOMIAL LOGISTIC REGRESSION
# ============================================================
#The purpose of Pipeline is to connect multiple machine-learning steps into a single sequential workflow.

logistic_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),

        (
            "classifier",
            LogisticRegression(
                max_iter=2000,
                random_state=42
            )
        )
    ]
)

# ============================================================
# 10. TRAIN MODEL
# ============================================================

logistic_model.fit(X_train, y_train)

# ============================================================
# 11. PREDICTION
# ============================================================

y_pred = logistic_model.predict(X_test)

# Probability of Placement = class 1
y_probability = logistic_model.predict_proba(X_test)[:, 1]

# ============================================================
# 12. MODEL EVALUATION
# ============================================================

accuracy = accuracy_score(y_test, y_pred)

print("\n============================================")
print("BINOMIAL LOGISTIC REGRESSION RESULTS")
print("============================================")

print("\nAccuracy:")
print(round(accuracy, 4))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_pred,
        target_names=["Not Placed", "Placed"]
    )
)

print("\nROC-AUC:")
print(round(roc_auc_score(y_test, y_probability), 4))


Dataset Shape: (50000, 31)

Columns:
['StudentID', 'Gender', 'City', 'CollegeTier', 'Stream', 'Specialisation', 'Hostel', 'HistoryOfBacklogs', 'SGPA_Sem1', 'SGPA_Sem2', 'SGPA_Sem3', 'SGPA_Sem4', 'SGPA_Sem5', 'SGPA_Sem6', 'SGPA_Sem7', 'SGPA_Sem8', 'CGPA', 'AttendancePercent', 'Internships', 'Projects', 'Workshops', 'Certifications', 'Publications', 'AptitudeTestScore', 'SoftSkillsRating', 'CodingTestScore', 'MockInterviewScore', 'ExtraCurricular', 'CGPA_Tier', 'PlacementStatus', 'IsAnomaly']


/tmp/ipykernel_1326/2892183833.py:127: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X[col] = X[col].fillna(X[col].median())
/tmp/ipykernel_1326/2892183833.py:130: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X[col] = X[col].fillna(X[col].mode()[0])



BINOMIAL LOGISTIC REGRESSION RESULTS

Accuracy:
0.9197

Confusion Matrix:
[[2926  503]
 [ 300 6271]]

Classification Report:
              precision    recall  f1-score   support

  Not Placed       0.91      0.85      0.88      3429
      Placed       0.93      0.95      0.94      6571

    accuracy                           0.92     10000
   macro avg       0.92      0.90      0.91     10000
weighted avg       0.92      0.92      0.92     10000


ROC-AUC:
0.9792


In [ ]:
#L1 regression
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    roc_auc_score
)

# ============================================================
# 1. LOAD DATASET
# ============================================================

df = pd.read_csv("/content/placement_predict_50k Dataset (1).csv")

print("Dataset Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())


# ============================================================
# 2. TARGET VARIABLE
# ============================================================

# PlacementStatus is already:
# 0 = Not Placed
# 1 = Placed

y = df["PlacementStatus"]


# ============================================================
# 3. SELECT PREDICTOR VARIABLES
# ============================================================

features = [
    "Gender",
    "City",
    "CollegeTier",
    "Stream",
    "Specialisation",
    "Hostel",
    "HistoryOfBacklogs",

    "SGPA_Sem1",
    "SGPA_Sem2",
    "SGPA_Sem3",
    "SGPA_Sem4",
    "SGPA_Sem5",
    "SGPA_Sem6",
    "SGPA_Sem7",
    "SGPA_Sem8",

    "CGPA",
    "AttendancePercent",

    "Internships",
    "Projects",
    "Workshops",
    "Certifications",
    "Publications",

    "AptitudeTestScore",
    "SoftSkillsRating",
    "CodingTestScore",
    "MockInterviewScore",
    "ExtraCurricular"
]

#X = df[features].copy()
X = df[features]

# ============================================================
# 4. CATEGORICAL VARIABLES
# ============================================================

categorical_features = [
    "Gender",
    "City",
    "CollegeTier",
    "Stream",
    "Specialisation",
    "Hostel",
    "HistoryOfBacklogs"
]


# ============================================================
# 5. NUMERICAL VARIABLES
# ============================================================

numerical_features = [
    "SGPA_Sem1",
    "SGPA_Sem2",
    "SGPA_Sem3",
    "SGPA_Sem4",
    "SGPA_Sem5",
    "SGPA_Sem6",
    "SGPA_Sem7",
    "SGPA_Sem8",

    "CGPA",
    "AttendancePercent",

    "Internships",
    "Projects",
    "Workshops",
    "Certifications",
    "Publications",

    "AptitudeTestScore",
    "SoftSkillsRating",
    "CodingTestScore",
    "MockInterviewScore",
    "ExtraCurricular"
]


# ============================================================
# 6. HANDLE MISSING VALUES
# ============================================================

for col in numerical_features:
    X[col] = X[col].fillna(X[col].median())

for col in categorical_features:
    X[col] = X[col].fillna(X[col].mode()[0])


# ============================================================
# 7. TRAIN-TEST SPLIT
# ============================================================
#stratify=y = Keep the same proportion of values in both training and testing splits.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)


# ============================================================
# 8. PREPROCESSING
# ============================================================
#ColumnTransformer allows us to preprocess different types of features differently and combine the results into a single feature matrix

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            numerical_features
        ),

        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore",
                drop="first"
            ),
            categorical_features
        )
    ]
)


# ============================================================
# 9. BINOMIAL LOGISTIC REGRESSION
# ============================================================
#The purpose of Pipeline is to connect multiple machine-learning steps into a single sequential workflow.

l1_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            LogisticRegression(
                penalty="l1",
                solver="liblinear",
                C=1.0,
                max_iter=2000,
                random_state=42
            )
        )
    ]
)

l1_model.fit(X_train, y_train)

y_pred_l1 = l1_model.predict(X_test)

y_prob_l1 = l1_model.predict_proba(X_test)[:, 1]


# ============================================================
# 10. TRAIN MODEL
# ============================================================

#logistic_model.fit(X_train, y_train)


# ============================================================
# 11. PREDICTION
# ============================================================

#y_pred = logistic_model.predict(X_test)

# Probability of Placement = class 1
#y_probability = logistic_model.predict_proba(X_test)[:, 1]


# ============================================================
# 12. MODEL EVALUATION
# ============================================================

accuracy = accuracy_score(y_test, y_pred)

print("\n============================================")
print("BINOMIAL LOGISTIC REGRESSION RESULTS")
print("============================================")

print("\nAccuracy:")
print(round(accuracy, 4))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_pred,
        target_names=["Not Placed", "Placed"]
    )
)

print("\nROC-AUC:")
print(round(roc_auc_score(y_test, y_probability), 4))

Dataset Shape: (50000, 31)

Columns:
['StudentID', 'Gender', 'City', 'CollegeTier', 'Stream', 'Specialisation', 'Hostel', 'HistoryOfBacklogs', 'SGPA_Sem1', 'SGPA_Sem2', 'SGPA_Sem3', 'SGPA_Sem4', 'SGPA_Sem5', 'SGPA_Sem6', 'SGPA_Sem7', 'SGPA_Sem8', 'CGPA', 'AttendancePercent', 'Internships', 'Projects', 'Workshops', 'Certifications', 'Publications', 'AptitudeTestScore', 'SoftSkillsRating', 'CodingTestScore', 'MockInterviewScore', 'ExtraCurricular', 'CGPA_Tier', 'PlacementStatus', 'IsAnomaly']


/tmp/ipykernel_1326/2121464789.py:132: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X[col] = X[col].fillna(X[col].median())
/tmp/ipykernel_1326/2121464789.py:135: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X[col] = X[col].fillna(X[col].mode()[0])



BINOMIAL LOGISTIC REGRESSION RESULTS

Accuracy:
0.9197

Confusion Matrix:
[[2926  503]
 [ 300 6271]]

Classification Report:
              precision    recall  f1-score   support

  Not Placed       0.91      0.85      0.88      3429
      Placed       0.93      0.95      0.94      6571

    accuracy                           0.92     10000
   macro avg       0.92      0.90      0.91     10000
weighted avg       0.92      0.92      0.92     10000


ROC-AUC:
0.9792


In [ ]:
#L2 regression
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    roc_auc_score
)

# ============================================================
# 1. LOAD DATASET
# ============================================================

df = pd.read_csv("/content/placement_predict_50k Dataset (1).csv")

print("Dataset Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())


# ============================================================
# 2. TARGET VARIABLE
# ============================================================

# PlacementStatus is already:
# 0 = Not Placed
# 1 = Placed

y = df["PlacementStatus"]


# ============================================================
# 3. SELECT PREDICTOR VARIABLES
# ============================================================

features = [
    "Gender",
    "City",
    "CollegeTier",
    "Stream",
    "Specialisation",
    "Hostel",
    "HistoryOfBacklogs",

    "SGPA_Sem1",
    "SGPA_Sem2",
    "SGPA_Sem3",
    "SGPA_Sem4",
    "SGPA_Sem5",
    "SGPA_Sem6",
    "SGPA_Sem7",
    "SGPA_Sem8",

    "CGPA",
    "AttendancePercent",

    "Internships",
    "Projects",
    "Workshops",
    "Certifications",
    "Publications",

    "AptitudeTestScore",
    "SoftSkillsRating",
    "CodingTestScore",
    "MockInterviewScore",
    "ExtraCurricular"
]

#X = df[features].copy()
X = df[features]

# ============================================================
# 4. CATEGORICAL VARIABLES
# ============================================================

categorical_features = [
    "Gender",
    "City",
    "CollegeTier",
    "Stream",
    "Specialisation",
    "Hostel",
    "HistoryOfBacklogs"
]


# ============================================================
# 5. NUMERICAL VARIABLES
# ============================================================

numerical_features = [
    "SGPA_Sem1",
    "SGPA_Sem2",
    "SGPA_Sem3",
    "SGPA_Sem4",
    "SGPA_Sem5",
    "SGPA_Sem6",
    "SGPA_Sem7",
    "SGPA_Sem8",

    "CGPA",
    "AttendancePercent",

    "Internships",
    "Projects",
    "Workshops",
    "Certifications",
    "Publications",

    "AptitudeTestScore",
    "SoftSkillsRating",
    "CodingTestScore",
    "MockInterviewScore",
    "ExtraCurricular"
]


# ============================================================
# 6. HANDLE MISSING VALUES
# ============================================================

for col in numerical_features:
    X[col] = X[col].fillna(X[col].median())

for col in categorical_features:
    X[col] = X[col].fillna(X[col].mode()[0])


# ============================================================
# 7. TRAIN-TEST SPLIT
# ============================================================
#stratify=y = Keep the same proportion of values in both training and testing splits.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)


# ============================================================
# 8. PREPROCESSING
# ============================================================
#ColumnTransformer allows us to preprocess different types of features differently and combine the results into a single feature matrix

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            numerical_features
        ),

        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore",
                drop="first"
            ),
            categorical_features
        )
    ]
)


# ============================================================
# 9. BINOMIAL LOGISTIC REGRESSION
# ============================================================
#The purpose of Pipeline is to connect multiple machine-learning steps into a single sequential workflow.

l2_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            LogisticRegression(
                penalty="l2",
                solver="lbfgs",
                C=1.0,
                max_iter=2000,
                random_state=42
            )
        )
    ]
)

l2_model.fit(X_train, y_train)

y_pred_l2 = l2_model.predict(X_test)

y_prob_l2 = l2_model.predict_proba(X_test)[:, 1]


# ============================================================
# 10. TRAIN MODEL
# ============================================================

#logistic_model.fit(X_train, y_train)


# ============================================================
# 11. PREDICTION
# ============================================================

#y_pred = logistic_model.predict(X_test)

# Probability of Placement = class 1
#y_probability = logistic_model.predict_proba(X_test)[:, 1]


# ============================================================
# 12. MODEL EVALUATION
# ============================================================

accuracy = accuracy_score(y_test, y_pred)

print("\n============================================")
print("BINOMIAL LOGISTIC REGRESSION RESULTS")
print("============================================")

print("\nAccuracy:")
print(round(accuracy, 4))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_pred,
        target_names=["Not Placed", "Placed"]
    )
)

print("\nROC-AUC:")
print(round(roc_auc_score(y_test, y_probability), 4))

Dataset Shape: (50000, 31)

Columns:
['StudentID', 'Gender', 'City', 'CollegeTier', 'Stream', 'Specialisation', 'Hostel', 'HistoryOfBacklogs', 'SGPA_Sem1', 'SGPA_Sem2', 'SGPA_Sem3', 'SGPA_Sem4', 'SGPA_Sem5', 'SGPA_Sem6', 'SGPA_Sem7', 'SGPA_Sem8', 'CGPA', 'AttendancePercent', 'Internships', 'Projects', 'Workshops', 'Certifications', 'Publications', 'AptitudeTestScore', 'SoftSkillsRating', 'CodingTestScore', 'MockInterviewScore', 'ExtraCurricular', 'CGPA_Tier', 'PlacementStatus', 'IsAnomaly']


/tmp/ipykernel_1326/3774680677.py:132: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X[col] = X[col].fillna(X[col].median())
/tmp/ipykernel_1326/3774680677.py:135: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X[col] = X[col].fillna(X[col].mode()[0])



BINOMIAL LOGISTIC REGRESSION RESULTS

Accuracy:
0.9197

Confusion Matrix:
[[2926  503]
 [ 300 6271]]

Classification Report:
              precision    recall  f1-score   support

  Not Placed       0.91      0.85      0.88      3429
      Placed       0.93      0.95      0.94      6571

    accuracy                           0.92     10000
   macro avg       0.92      0.90      0.91     10000
weighted avg       0.92      0.92      0.92     10000


ROC-AUC:
0.9792


In [ ]:
#Elastic-net regularization
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    roc_auc_score
)

# ============================================================
# 1. LOAD DATASET
# ============================================================

df = pd.read_csv("/content/placement_predict_50k Dataset (1).csv")

print("Dataset Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())


# ============================================================
# 2. TARGET VARIABLE
# ============================================================

# PlacementStatus is already:
# 0 = Not Placed
# 1 = Placed

y = df["PlacementStatus"]


# ============================================================
# 3. SELECT PREDICTOR VARIABLES
# ============================================================

features = [
    "Gender",
    "City",
    "CollegeTier",
    "Stream",
    "Specialisation",
    "Hostel",
    "HistoryOfBacklogs",

    "SGPA_Sem1",
    "SGPA_Sem2",
    "SGPA_Sem3",
    "SGPA_Sem4",
    "SGPA_Sem5",
    "SGPA_Sem6",
    "SGPA_Sem7",
    "SGPA_Sem8",

    "CGPA",
    "AttendancePercent",

    "Internships",
    "Projects",
    "Workshops",
    "Certifications",
    "Publications",

    "AptitudeTestScore",
    "SoftSkillsRating",
    "CodingTestScore",
    "MockInterviewScore",
    "ExtraCurricular"
]

#X = df[features].copy()
X = df[features]

# ============================================================
# 4. CATEGORICAL VARIABLES
# ============================================================

categorical_features = [
    "Gender",
    "City",
    "CollegeTier",
    "Stream",
    "Specialisation",
    "Hostel",
    "HistoryOfBacklogs"
]


# ============================================================
# 5. NUMERICAL VARIABLES
# ============================================================

numerical_features = [
    "SGPA_Sem1",
    "SGPA_Sem2",
    "SGPA_Sem3",
    "SGPA_Sem4",
    "SGPA_Sem5",
    "SGPA_Sem6",
    "SGPA_Sem7",
    "SGPA_Sem8",

    "CGPA",
    "AttendancePercent",

    "Internships",
    "Projects",
    "Workshops",
    "Certifications",
    "Publications",

    "AptitudeTestScore",
    "SoftSkillsRating",
    "CodingTestScore",
    "MockInterviewScore",
    "ExtraCurricular"
]


# ============================================================
# 6. HANDLE MISSING VALUES
# ============================================================

for col in numerical_features:
    X[col] = X[col].fillna(X[col].median())

for col in categorical_features:
    X[col] = X[col].fillna(X[col].mode()[0])


# ============================================================
# 7. TRAIN-TEST SPLIT
# ============================================================
#stratify=y = Keep the same proportion of values in both training and testing splits.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)


# ============================================================
# 8. PREPROCESSING
# ============================================================
#ColumnTransformer allows us to preprocess different types of features differently and combine the results into a single feature matrix

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            numerical_features
        ),

        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore",
                drop="first"
            ),
            categorical_features
        )
    ]
)


# ============================================================
# 9. BINOMIAL LOGISTIC REGRESSION
# ============================================================
#The purpose of Pipeline is to connect multiple machine-learning steps into a single sequential workflow.

elastic_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            LogisticRegression(
                penalty="elasticnet",
                solver="saga",
                l1_ratio=0.5,
                C=1.0,
                max_iter=5000,
                random_state=42
            )
        )
    ]
)

elastic_model.fit(X_train, y_train)

y_pred_elastic = elastic_model.predict(X_test)

y_prob_elastic = elastic_model.predict_proba(X_test)[:, 1]


# ============================================================
# 10. TRAIN MODEL
# ============================================================

#logistic_model.fit(X_train, y_train)


# ============================================================
# 11. PREDICTION
# ============================================================

#y_pred = logistic_model.predict(X_test)

# Probability of Placement = class 1
#y_probability = logistic_model.predict_proba(X_test)[:, 1]


# ============================================================
# 12. MODEL EVALUATION
# ============================================================

accuracy = accuracy_score(y_test, y_pred)

print("\n============================================")
print("BINOMIAL LOGISTIC REGRESSION RESULTS")
print("============================================")

print("\nAccuracy:")
print(round(accuracy, 4))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_pred,
        target_names=["Not Placed", "Placed"]
    )
)

print("\nROC-AUC:")
print(round(roc_auc_score(y_test, y_probability), 4))

Dataset Shape: (50000, 31)

Columns:
['StudentID', 'Gender', 'City', 'CollegeTier', 'Stream', 'Specialisation', 'Hostel', 'HistoryOfBacklogs', 'SGPA_Sem1', 'SGPA_Sem2', 'SGPA_Sem3', 'SGPA_Sem4', 'SGPA_Sem5', 'SGPA_Sem6', 'SGPA_Sem7', 'SGPA_Sem8', 'CGPA', 'AttendancePercent', 'Internships', 'Projects', 'Workshops', 'Certifications', 'Publications', 'AptitudeTestScore', 'SoftSkillsRating', 'CodingTestScore', 'MockInterviewScore', 'ExtraCurricular', 'CGPA_Tier', 'PlacementStatus', 'IsAnomaly']


/tmp/ipykernel_1326/4080155175.py:132: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X[col] = X[col].fillna(X[col].median())
/tmp/ipykernel_1326/4080155175.py:135: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X[col] = X[col].fillna(X[col].mode()[0])



BINOMIAL LOGISTIC REGRESSION RESULTS

Accuracy:
0.9197

Confusion Matrix:
[[2926  503]
 [ 300 6271]]

Classification Report:
              precision    recall  f1-score   support

  Not Placed       0.91      0.85      0.88      3429
      Placed       0.93      0.95      0.94      6571

    accuracy                           0.92     10000
   macro avg       0.92      0.90      0.91     10000
weighted avg       0.92      0.92      0.92     10000


ROC-AUC:
0.9792


In [ ]:
# ============================================================
# PREDICT PLACEMENT FOR A NEW STUDENT
# ============================================================

new_student = pd.DataFrame([{
    "Gender": "Male",
    "City": "Bangalore",
    "CollegeTier": 1,
    "Stream": "Computer Science",
    "Specialisation": "Computer Science",
    "Hostel": "Yes",
    "HistoryOfBacklogs": "No",

    "SGPA_Sem1": 8.2,
    "SGPA_Sem2": 8.4,
    "SGPA_Sem3": 8.5,
    "SGPA_Sem4": 8.6,
    "SGPA_Sem5": 8.7,
    "SGPA_Sem6": 8.8,
    "SGPA_Sem7": 8.9,
    "SGPA_Sem8": 9.0,

    "CGPA": 8.65,
    "AttendancePercent": 92,

    "Internships": 2,
    "Projects": 4,
    "Workshops": 5,
    "Certifications": 4,
    "Publications": 1,

    "AptitudeTestScore": 85,
    "SoftSkillsRating": 8.5,
    "CodingTestScore": 88,
    "MockInterviewScore": 8.2,
    "ExtraCurricular": 4
}])


# ============================================================
# PREDICTION
# ============================================================

prediction = elastic_model.predict(new_student)[0]

probability = elastic_model.predict_proba(
    new_student
)[0, 1]


# ============================================================
# DISPLAY RESULT
# ============================================================

print("\n============================================")
print("NEW STUDENT PLACEMENT PREDICTION")
print("============================================")

if prediction == 1:
    print("Predicted Placement Status : PLACED")
else:
    print("Predicted Placement Status : NOT PLACED")

print(
    f"Probability of Placement    : {probability * 100:.2f}%"
)

print(
    f"Probability of Not Placed   : {(1 - probability) * 100:.2f}%"
)

TypeError: ufunc 'isnan' not supported for the input types, and the inputs could not be safely coerced to any supported types according to the casting rule ''safe''